# 실험 04: 토큰(Token), 제어, 비용 최적화

## 1. 실험 제목과 목표
- **제목**: Tiktoken 인코딩 관찰 및 Context Limit(토큰 한도) 대처 실험
- **목표**: 텍스트가 어떻게 토큰으로 분해되는지 눈으로 확인하고, `max_tokens`의 Hard cutoff 현상을 체험합니다. 나아가 프롬프트 엔지니어링으로 이를 해결하고, 실제 호출 비용을 계산해보며 Token 관리에 대한 감각을 기릅니다.

## 2. 실험 개요
1. **비교 실험 1**: 영어, 한국어, 프로그래밍 코드의 토큰 소비량 효율 비교
2. **비교 실험 2**: `max_tokens`를 매우 작게 주었을 때의 문장 짤림(`finish_reason: length`) 현상 관찰
3. **비교 실험 3**: Prompt Engineering으로 짤림 없이 강제 요약 시키기
4. **비교 실험 4**: API 응답의 `usage` 객체를 활용한 비용 계산 (gpt-4o-mini 기준)
5. **실패 케이스**: LLM이 받아들일 수 있는 최대 입력 토큰 한계(Context Window)를 초과했을 때의 붕괴 시연

## 3. 라이브러리 import

In [ ]:
import os
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

## 4. 환경 변수 로드 및 client 생성

In [ ]:
load_dotenv()
client = OpenAI()

## 5. 비교 실험 1: Tiktoken 인코딩 효율성 비교 (En vs Ko vs Code)
동일한 의미를 지닌 텍스트가 언어에 따라 얼마나 다르게 쪼개지는지 확인합니다.

In [ ]:
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

text_en = "Hello, how are you today? Let's write some code."
text_ko = "안녕하세요, 오늘 하루 어떠신가요? 우리 코드를 좀 작성해봅시다."
text_code = "def greet():\n    print('Hello World')"

def print_token_info(label, text):
    tokens = encoding.encode(text)
    # 디코딩해서 각 토큰이 어떻게 생겼는지 시각적으로 확인
    token_chunks = [encoding.decode([t]) for t in tokens]
    print(f"[{label}] 글자 수: {len(text)} -> 토큰 수: {len(tokens)}")
    print(f"조각 확인: {token_chunks}\n")

print_token_info("영어", text_en)
print_token_info("한국어", text_ko)
print_token_info("코드", text_code)

print("-> 결과: 영어와 파이썬 코드는 매우 효율적(1단어~1토큰)으로 쪼개지지만, 한국어는 자모음이나 조사 단위로 잘게 쪼개져 토큰(비용) 소모가 더 큽니다.")

## 6. 비교 실험 2: max_tokens 제한으로 인한 강제 잘림 현상
모델의 출력을 제어하기 위해 `max_tokens`를 작게 주면 어떻게 되는지 봅니다.

In [ ]:
prompt_history = "한국의 역사에 대해 설명해줘."

res_limit = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_history}],
    max_tokens=20  # 최대 토큰 수를 20개로 억제
)

print("[잘린 응답]")
print(res_limit.choices[0].message.content)
print("\n종료 사유 (finish_reason):", res_limit.choices[0].finish_reason)
print("-> 결과: 문장 중간에 가차없이 잘려버리며, 'length'라는 사유로 강제 종료됩니다.")

## 7. 비교 실험 3: 프롬프트 엔지니어링으로 잘림 방지
토큰을 아끼면서도 온전한 문장을 받으려면 System Prompt로 길이를 강제해야 합니다.

In [ ]:
res_guided = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "당신은 요약의 달인입니다. 반드시 한 문장, 15단어 이내로 답변하세요."},
        {"role": "user", "content": prompt_history}
    ],
    max_tokens=40  # 여유 공간 제공
)

print("[안전한 응답]")
print(res_guided.choices[0].message.content)
print("\n종료 사유 (finish_reason):", res_guided.choices[0].finish_reason)
print("-> 결과: 제한된 범위 내에서 스스로 마침표를 찍으며 'stop' 사유로 정상 종료되었습니다.")

## 8. 비교 실험 4: API 사용량(Usage)을 통한 비용 계산
gpt-4o-mini 모델의 단가(예: 입력 1M 토큰당 $0.15, 출력 1M 토큰당 $0.60)를 기준으로 1회 요청 비용을 계산해 봅니다.

In [ ]:
usage = res_guided.usage

input_tokens = usage.prompt_tokens
output_tokens = usage.completion_tokens
total_tokens = usage.total_tokens

# 가상의 gpt-4o-mini 달러 단가 (가격 정책에 따라 변동 가능)
input_price_per_1m = 0.15 
output_price_per_1m = 0.60

cost_input = (input_tokens / 1000000) * input_price_per_1m
cost_output = (output_tokens / 1000000) * output_price_per_1m
total_cost_usd = cost_input + cost_output

print("[토큰 사용량]")
print(f"- 프롬프트(입력) 토큰: {input_tokens}")
print(f"- 답변(출력) 토큰: {output_tokens}")
print(f"- 총합: {total_tokens}")
print(f"\n[예상 비용]: ${total_cost_usd:.6f}")

## 9. 실패/주의 케이스: Context Window 초과 (BadRequest)
모델이 처리할 수 있는 최대 글자 수(보통 128k 토큰)를 고의로 넘어보겠습니다.

In [ ]:
print("[거대한 프롬프트 주입]")
massive_prompt = "A " * 200000  # 약 20만 개의 단어로 토큰 한계를 넘기기

try:
    client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": massive_prompt}]
    )
except Exception as e:
    print("🔥 에러 발생:", type(e).__name__)
    print("에러 메시지:", str(e)[:200] + "...")
    print("\n-> 결과: 입력 길이 초과로 API가 즉시 거절(BadRequest)합니다. RAG 등에서 문서 전체를 때려 넣을 때 흔히 겪는 문제입니다.")

## 10. 결과 해석

1. **한국어 토큰 패널티**: 영어 서비스 대비 한국어 서비스는 동일한 내용을 출력해도 Token(비용)이 1.5배~2배 정도 비쌉니다. 
2. **길이 제어**: `max_tokens`는 기계적인 '가위' 역할을 하므로, 잘림을 방지하려면 System Prompt라는 '가이드'가 동반되어야 완벽한 제어가 가능합니다.
3. **Context Window의 벽**: LLM은 입력 길이가 무한하지 않습니다. 128k 토큰 등을 넘어가면 요청 자체가 에러를 반환하므로, 긴 문서는 쪼개서 넣거나(Chunking, RAG) 요약해야 합니다.

## 11. Notion 실험 로그

### 발견한 문제점
* 한국어로 개발할 때 영문 서비스보다 예상치 못하게 API 비용이 더 빠르게 소진될 수 있음을 `tiktoken`을 통해 눈으로 확인함.
* `max_tokens`만 믿고 출력 길이를 억제하면 사용자 화면에 중간에 잘려버린 문장이 나가게 되어 치명적인 버그처럼 보임.

### 다음 개선 방향
* 한국어 토큰 소모를 줄이거나 속도를 높이기 위한 캐싱(Caching) 전략이나, 문서를 잘게 쪼개는 Text Splitter(랭체인) 개념을 다음 단계에서 도입해야 함.
* 모델의 응답에 대해 항상 `finish_reason == 'stop'` 인지 검증하는 로직을 서비스에 추가해야 함.